# Проект: pretrain и SFT для LLM

**Цель:** реализовать pretrain и posttrain этапы обучения LLM

**Данные:** тексты произведений русской литературы

## Настройка окружения

In [3]:
from pathlib import Path
import os
import random
import re

import torch
from datasets import Dataset, load_dataset

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_DIR = Path.cwd()
DATASET_PATH = PROJECT_DIR/'dataset'/'corpus'
PRETRAIN_DIR = PROJECT_DIR/'pretrain_artifacts'
TOKENIZER_DIR = PRETRAIN_DIR/'tokenizer_bpe'
TOKENIZER_JSON = PRETRAIN_DIR/'tokenizer_bpe.json'
PRETRAIN_MODEL_DIR = PROJECT_DIR/'model_pretrain_lit_ru'
SFT_MODEL_DIR = PROJECT_DIR/'model_sft_qwen_ru'

VOCAB_SIZE = 3000
CONTEXT_LENGTH = 512
PRETRAIN_EFFECTIVE_BATCH_SIZE = 64

PRETRAIN_DIR.mkdir(parents=True, exist_ok=True)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

CUDA: True
Tesla T4


# Pretrain

## 1. Загрузка и упаковка корпуса

In [2]:
txt_files = sorted(DATASET_PATH.rglob('*.txt'))

raw_texts = [path.read_text(encoding='utf-8') for path in txt_files]


print(f'Файлов: {len(txt_files)}')
print(f'Символов до очистки: {sum(len(t) for t in raw_texts):,}')

Файлов: 108
Символов до очистки: 45,348,575


## 2. Препроцессинг

Очистка включает удаление дублей, фильтрацию предложений с латиницей и другими некириллическими буквами, нормализацию повторяющейся пунктуации и пробелов, нарезку на чанки.

In [3]:
# очистка
CYRILLIC_RE = re.compile(r'[а-яё]', re.IGNORECASE)
FOREIGN_LETTER_RE = re.compile(r'[a-zA-ZÀ-ÖØ-öø-ÿĀ-žΑ-ω]')
SENTENCE_SPLIT_RE = re.compile(r'(?<=[.!?])\s+')


def normalize_text(text: str) -> str:
    text = text.replace('\ufeff', ' ').replace('\xa0', ' ')
    text = text.replace('—', ' - ').replace('–', ' - ')
    text = text.replace('«', '\'').replace('»', '\'')
    text = text.lower()
    text = re.sub(r'[^а-яё0-9\s.,!?;:\'()\-]', ' ', text)
    text = re.sub(r'([.!?]){2,}', r'\1', text)
    text = re.sub(r'([,;:]){2,}', r'\1', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def is_good_sentence(sentence: str) -> bool:
    sentence = sentence.strip()
    if len(sentence) < 20:
        return False
    if not CYRILLIC_RE.search(sentence):
        return False
    if FOREIGN_LETTER_RE.search(sentence):
        return False
    return True


seen = set()
sentences = []
for text in raw_texts:
    text = text.replace('\ufeff', ' ').replace('\xa0', ' ')
    text = re.sub(r'\s+', ' ', text)
    for raw_sent in SENTENCE_SPLIT_RE.split(text):
        raw_sent = raw_sent.strip(' -\t\n\r')
        # Сначала отбрасываем предложения с буквами не из кириллицы, затем чистим пунктуацию.
        if not is_good_sentence(raw_sent):
            continue
        sent = normalize_text(raw_sent).strip(' -\t\n\r')
        if is_good_sentence(sent) and sent not in seen:
            seen.add(sent)
            sentences.append(sent)

print(f'Предложений после очистки и дедупликации: {len(sentences):,}')
print(sentences[0])

Предложений после очистки и дедупликации: 427,811
посвящается любови евгеньевне белозерской пошел мелкий снег и вдруг повалил хлопьями.


In [4]:
# чанки
def make_text_chunks(sentences, max_chars=1800):
    chunks = []
    current = []
    current_len = 0
    for sent in sentences:
        add_len = len(sent) + 1
        if current and current_len + add_len > max_chars:
            chunks.append(' '.join(current))
            current = []
            current_len = 0
        current.append(sent)
        current_len += add_len
    if current:
        chunks.append(' '.join(current))
    return chunks


text_chunks = make_text_chunks(sentences)
random.shuffle(text_chunks)

corpus_for_tokenizer = PRETRAIN_DIR/'clean_corpus.txt'
corpus_for_tokenizer.write_text('\n'.join(text_chunks), encoding='utf-8')

print(f'Текстовых чанков: {len(text_chunks):,}')
print(f'Символов после очистки: {sum(len(c) for c in text_chunks):,}')

Текстовых чанков: 23,185
Символов после очистки: 39,847,365


## 3. Обучение собственного BPE-токенизатора

In [5]:
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, processors
from tokenizers.trainers import BpeTrainer
from transformers import PreTrainedTokenizerFast

SPECIAL_TOKENS = ['<unk>', '<pad>', '<bos>', '<eos>']

bpe_tokenizer = Tokenizer(models.BPE(unk_token='<unk>'))
bpe_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
bpe_tokenizer.decoder = decoders.BPEDecoder()

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    end_of_word_suffix='</w>',
    special_tokens=SPECIAL_TOKENS,
)

bpe_tokenizer.train([str(corpus_for_tokenizer)], trainer=trainer)
bpe_tokenizer.post_processor = processors.TemplateProcessing(
    single='<bos> $A <eos>',
    special_tokens=[
        ('<bos>', bpe_tokenizer.token_to_id('<bos>')),
        ('<eos>', bpe_tokenizer.token_to_id('<eos>')),
    ],
)
bpe_tokenizer.save(str(TOKENIZER_JSON))

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=str(TOKENIZER_JSON),
    unk_token='<unk>',
    pad_token='<pad>',
    bos_token='<bos>',
    eos_token='<eos>',
)
tokenizer.model_max_length = CONTEXT_LENGTH
tokenizer.save_pretrained(TOKENIZER_DIR)

print('Размер словаря:', len(tokenizer))
print(tokenizer.tokenize('федор михайлович достоевский родился в москве в 1821 году'))




Размер словаря: 3000
['федо', 'р</w>', 'михай', 'ло', 'вич</w>', 'досто', 'е', 'вский</w>', 'роди', 'лся</w>', 'в</w>', 'моск', 'ве</w>', 'в</w>', '1', '8', '2', '1</w>', 'го', 'ду</w>']


## 4. Токенизация и подготовка `Dataset` для pretrain

In [6]:
def encode_chunk(text):
    encoded = tokenizer(
        text,
        add_special_tokens=True,
        truncation=True,
        max_length=CONTEXT_LENGTH,
        padding='max_length',
    )
    encoded['input_ids'][-1] = tokenizer.eos_token_id
    return {'input_ids': encoded['input_ids'], 'attention_mask': encoded['attention_mask']}


encoded_rows = [encode_chunk(chunk) for chunk in text_chunks]
dataset = Dataset.from_list(encoded_rows).train_test_split(test_size=0.02, seed=SEED)
train_dataset = dataset['train']
eval_dataset = dataset['test']

print(train_dataset)
print(eval_dataset)
print(tokenizer.decode(train_dataset[0]['input_ids'][:80]))

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 22721
})
Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 464
})
<bos>уж конечно , роман нравственный ! вскричала с злобною усмешкою полицеймейстерша . эти смиренницы любят выставлять напоказ добродетели , которых у них не водится . да почему же вы полагаете в ней скрытые пороки ? произнес голос из толпы . я знаю давно мадам гольц


## 5. Инициализация decoder-only модели

In [7]:
from transformers import LlamaConfig, LlamaForCausalLM, DataCollatorForLanguageModeling

config = LlamaConfig(
    vocab_size=len(tokenizer),
    hidden_size=1024,
    intermediate_size=1536,
    num_hidden_layers=16,
    num_attention_heads=16,
    num_key_value_heads=8,
    max_position_embeddings=CONTEXT_LENGTH,
    rms_norm_eps=1e-5,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

model = LlamaForCausalLM(config)
model.gradient_checkpointing_enable()
n_params = sum(p.numel() for p in model.parameters())
print(f'Параметров: {n_params/1e6:.1f}M')

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Параметров: 132.0M


In [8]:
from transformers import TrainerCallback

test_prompts = [
    'Все мысли, которые имеют огромные последствия',
    'Сила войска зависит от его духа',
    'Мысль о том, что он принес страдания',
    'Человек сознает себя свободным',
    'Что бы ни случилось, я всегда буду',
    'Любовь мешает смерти',
    'Нет, жизнь не кончена',
    'Всякая мысль, даже самая простая',
    'Война не любезность, а самое гадкое дело',
    'Чтобы жить честно',
]


def generate_pretrain_samples(model, tokenizer, prompts, max_new_tokens=80):
    device = model.device
    model.eval()
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors='pt').to(device)
        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.8,
                top_p=0.9,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        print('PROMPT:', prompt)
        print(tokenizer.decode(output_ids[0], skip_special_tokens=True))
        print('-' * 80)
    model.train()


class PromptGenerationCallback(TrainerCallback):
    def __init__(self, prompts, every_n_steps=1000):
        self.prompts = prompts
        self.every_n_steps = every_n_steps

    def on_step_end(self, args, state, control, model=None, tokenizer=None, processing_class=None, **kwargs):
        if state.global_step and state.global_step % self.every_n_steps == 0:
            print(f'\n=== Generation at step {state.global_step} ===')
            active_tokenizer = tokenizer or processing_class or kwargs.get('processing_class')
            if model is not None and active_tokenizer is not None:
                generate_pretrain_samples(model, active_tokenizer, self.prompts[:3], max_new_tokens=64)
        return control

## 6. Обучение pretrain через `Trainer`

In [9]:
from inspect import signature
from transformers import Trainer, TrainingArguments


def build_args(args_class, **kwargs):
    params = signature(args_class.__init__).parameters
    if 'eval_strategy' in kwargs and 'eval_strategy' not in params and 'evaluation_strategy' in params:
        kwargs['evaluation_strategy'] = kwargs.pop('eval_strategy')
    filtered_kwargs = {key: value for key, value in kwargs.items() if key in params}
    return args_class(**filtered_kwargs)


def build_trainer(**kwargs):
    params = signature(Trainer.__init__).parameters
    if 'processing_class' in kwargs and 'processing_class' not in params and 'tokenizer' in params:
        kwargs['tokenizer'] = kwargs.pop('processing_class')
    filtered_kwargs = {key: value for key, value in kwargs.items() if key in params}
    return Trainer(**filtered_kwargs)


per_device_train_batch_size = 4
gradient_accumulation_steps = PRETRAIN_EFFECTIVE_BATCH_SIZE//per_device_train_batch_size

training_args = build_args(
    TrainingArguments,
    output_dir=str(PRETRAIN_MODEL_DIR),
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=2e-4,
    warmup_steps=10,
    weight_decay=0.1,
    lr_scheduler_type='cosine',
    logging_steps=50,
    eval_strategy='steps',
    eval_steps=500,
    save_steps=1000,
    save_total_limit=2,
    bf16=torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8,
    fp16=torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] < 8,
    gradient_checkpointing=True,
    optim='adamw_torch_fused' if torch.cuda.is_available() else 'adamw_torch',
    report_to='none',
    remove_unused_columns=False,
)

trainer = build_trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[PromptGenerationCallback(test_prompts, every_n_steps=1000)],
)

trainer.train()
trainer.save_model(PRETRAIN_MODEL_DIR)
tokenizer.save_pretrained(PRETRAIN_MODEL_DIR)


Step,Training Loss,Validation Loss
500,3.861997,3.864492
1000,3.476446,3.632300
1068,3.479656,3.631182



=== Generation at step 1000 ===
PROMPT: Все мысли, которые имеют огромные последствия
се мысли , которые имеют огромные последствия к нему ( хотя он , как это , в конце концов , был не только для него ), но и когда он уже выходил из гостиной , как в городе , - и теперь , появившись в москве , он с тем же отвращением подал ему эту записку , в которой
--------------------------------------------------------------------------------
PROMPT: Сила войска зависит от его духа
ила войска зависит от его духа , и в самом деле : а что , как поступить , солому с горя ? да , если не доверяли , то хоть и вылезай , - все это оправдание ! а теперь и с той же силой ! она только прибегала к себе ,
--------------------------------------------------------------------------------
PROMPT: Мысль о том, что он принес страдания
ысль о том , что он принес страдания с того же секунды , как он изъявил эту речь , и даже когда его увезли за то , что он был в петербурге , и , признавая то , что было , и все эти люди

Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.39s/it]


('/home/ubuntu/project/model_pretrain_lit_ru/tokenizer_config.json',
 '/home/ubuntu/project/model_pretrain_lit_ru/tokenizer.json')

## 7. Финальная генерация pretrain-модели

In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer

pretrain_tokenizer = AutoTokenizer.from_pretrained(PRETRAIN_MODEL_DIR)
pretrain_model = AutoModelForCausalLM.from_pretrained(PRETRAIN_MODEL_DIR, device_map='auto')

generate_pretrain_samples(pretrain_model, pretrain_tokenizer, test_prompts, max_new_tokens=96)

del pretrain_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Loading weights: 100%|██████████| 147/147 [00:00<00:00, 1877.25it/s]


PROMPT: Все мысли, которые имеют огромные последствия
се мысли , которые имеют огромные последствия в жизни ( как ни странно было уложить в этой комнате ), когда , наконец , с усилием вызывая все это время от разврата , она наводила на них свои пространства . она не могла найти никакого намерения к ней , и не мог сделать их . кроме того , что она говорила , что он не может понять , что ее поступки и не может быть увлекательны , и что она , кажется , ничего не
--------------------------------------------------------------------------------
PROMPT: Сила войска зависит от его духа
ила войска зависит от его духа с женой и , в которой , когда он , задумавшись от злобы , уведомил меня , что я , по отъезде , выбегала из своей особую жизнь . он был небольшим , в который мы говорили о том , как только погубила его , но в этот раз он не мог понять того , что она ему выслушала . теперь , во всем его предчувствие , ему было приятно
------------------------------------------------------------------

# Post-train SFT

## 8. Базовая генерация Qwen

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL_NAME = 'Qwen/Qwen2.5-0.5B'
questions_rus = [
    'сколько планет в нашей солнечной системе?',
    'расскажи стих',
    'когда собирать крыжовник?',
    'Как быстро выучить новый язык?',
]


def build_qwen_prompt(tokenizer, question):
    messages = [
        {'role': 'system', 'content': 'Ты полезный русскоязычный ассистент.'},
        {'role': 'user', 'content': question},
    ]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except ImportError:
        return (
            '<|im_start|>system\nТы полезный русскоязычный ассистент.<|im_end|>\n'
            f'<|im_start|>user\n{question}<|im_end|>\n'
            '<|im_start|>assistant\n'
        )


def generate_answers(model, tokenizer, questions, max_new_tokens=220):
    model.eval()
    for i, question in enumerate(questions, 1):
        prompt = build_qwen_prompt(tokenizer, question)
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.05,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = output_ids[0, inputs['input_ids'].shape[1]:]
        answer = tokenizer.decode(generated, skip_special_tokens=True).strip()
        print(f'Model Input {i}:')
        print(question)
        print(f'Model Output {i}:')
        print(answer)
        print('=' * 80)


base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype='auto',
    device_map='auto',
    trust_remote_code=True,
)

generate_answers(base_model, base_tokenizer, questions_rus)

del base_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2039.67it/s]


Model Input 1:
сколько планет в нашей солнечной системе?
Model Output 1:
78
얏
сколько солнечных систем в нашей солнечной системе?얏
هائي
сколько планет на нашей солнечной системе?هائي
هائي
сколько солнечных систем в нашей солнечной системе?هائي
هائي
сколько планет на нашей солнечной системе?هائي
هائي
сколько солнечных систем в нашей солнечной системе?هائي
هائي
сколько планет на нашей солнечной системе?هائي
هائي
сколько солнечных систем в нашей солнечной системе?هائي
هائي
сколько планет на нашей солнечной системе?هائي
هائي
сколько солнечных систем в нашей солнечной системе?هائي
هائي
сколько планет на нашей солнечной системе?هائي
هائي
сколько солнечных систем в нашей солнечной системе?هائي
هائي
сколько планет на нашей солнечной системе?هائي
هائي
сколько планет на нашей
Model Input 2:
расскажи стих
Model Output 2:
Что ты умеешь?yre
WindowText
WindowText
WindowText
WindowText
WindowText
WindowText
WindowText
WindowText
WindowText
WindowText
WindowText
WindowText
WindowText
WindowText
Window

## 9. Подготовка `d0rj/alpaca-cleaned-ru` 

In [6]:
alpaca = load_dataset('d0rj/alpaca-cleaned-ru', split='train')
alpaca = alpaca.shuffle(seed=SEED)

SYSTEM_PROMPT = 'Ты полезный русскоязычный ассистент. Отвечай ясно, структурно и по-русски.'


def to_messages(example):
    user_parts = []
    instruction = (example.get('instruction') or '').strip()
    input_text = (example.get('input') or '').strip()
    if instruction:
        user_parts.append(instruction)
    if input_text:
        user_parts.append(input_text)
    user_text = '\n\n'.join(user_parts).strip()
    assistant_text = (example.get('output') or '').strip()
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_text},
        {'role': 'assistant', 'content': assistant_text},
    ]
    try:
        text = base_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    except ImportError:
        text = (
            f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
            f'<|im_start|>user\n{user_text}<|im_end|>\n'
            f'<|im_start|>assistant\n{assistant_text}<|im_end|>\n'
        )
    return {'text': text}


sft_dataset = alpaca.map(to_messages, remove_columns=alpaca.column_names)
sft_dataset = sft_dataset.filter(lambda x: len(x['text']) > 0)
split_sft = sft_dataset.train_test_split(test_size=0.02, seed=SEED)

print(split_sft)
print(split_sft['train'][0]['text'][:1200])

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 50724
    })
    test: Dataset({
        features: ['text'],
        num_rows: 1036
    })
})
<|im_start|>system
Ты полезный русскоязычный ассистент. Отвечай ясно, структурно и по-русски.<|im_end|>
<|im_start|>user
Составьте правильное вступительное заявление для речи о важности голосования.<|im_end|>
<|im_start|>assistant
Дамы и господа, уважаемые гости, для меня большая честь присутствовать здесь сегодня, чтобы обсудить тему, которая имеет основополагающее значение для нашего общества и функционирования нашей демократии, - важность осуществления нашего права голоса.<|im_end|>



## 11. SFT 

In [7]:
from inspect import signature

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig, prepare_model_for_kbit_training


def build_sft_args(**kwargs):
    params = signature(SFTConfig.__init__).parameters

    if 'eval_strategy' in kwargs and 'eval_strategy' not in params and 'evaluation_strategy' in params:
        kwargs['evaluation_strategy'] = kwargs.pop('eval_strategy')

    filtered_kwargs = {key: value for key, value in kwargs.items() if key in params}
    return SFTConfig(**filtered_kwargs)


def build_sft_trainer(**kwargs):
    params = signature(SFTTrainer.__init__).parameters

    if 'processing_class' in kwargs and 'processing_class' not in params and 'tokenizer' in params:
        kwargs['tokenizer'] = kwargs.pop('processing_class')

    filtered_kwargs = {key: value for key, value in kwargs.items() if key in params}
    return SFTTrainer(**filtered_kwargs)


sft_tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_NAME,
    trust_remote_code=True,
)

if sft_tokenizer.pad_token is None:
    sft_tokenizer.pad_token = sft_tokenizer.eos_token


sft_use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
sft_use_fp16 = torch.cuda.is_available() and not sft_use_bf16
sft_compute_dtype = torch.bfloat16 if sft_use_bf16 else torch.float16


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=sft_compute_dtype,
    bnb_4bit_use_double_quant=True,
)


sft_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=sft_compute_dtype,
    device_map='auto',
    trust_remote_code=True,
)

sft_model.config.use_cache = False
sft_model = prepare_model_for_kbit_training(sft_model)

if sft_use_fp16:
    for param in sft_model.parameters():
        if param.requires_grad and param.dtype == torch.bfloat16:
            param.data = param.data.to(torch.float16)


lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=[
        'q_proj',
        'k_proj',
        'v_proj',
        'o_proj',
        'gate_proj',
        'up_proj',
        'down_proj',
    ],
)


sft_args = build_sft_args(
    output_dir=str(SFT_MODEL_DIR),
    overwrite_output_dir=True,

    dataset_text_field='text',
    max_length=256,
    packing=False,

    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    warmup_steps=10,
    weight_decay=0.01,
    lr_scheduler_type='cosine',

    logging_steps=10,
    eval_strategy='no',

    save_steps=200,
    save_total_limit=1,

    bf16=sft_use_bf16,
    fp16=sft_use_fp16,

    gradient_checkpointing=True,
    optim='paged_adamw_8bit',

    report_to='none',
)


sft_train_dataset = (
    split_sft['train'].remove_columns('messages')
    if 'messages' in split_sft['train'].column_names
    else split_sft['train']
)

sft_eval_dataset = (
    split_sft['test'].remove_columns('messages')
    if 'messages' in split_sft['test'].column_names
    else split_sft['test']
)


sft_train_dataset = sft_train_dataset.select(
    range(min(300, len(sft_train_dataset)))
)

sft_eval_dataset = sft_eval_dataset.select(
    range(min(50, len(sft_eval_dataset)))
)


print("Train examples:", len(sft_train_dataset))
print("Eval examples:", len(sft_eval_dataset))
print("fp16:", sft_args.fp16)
print("bf16:", sft_args.bf16)


sft_trainer = build_sft_trainer(
    model=sft_model,
    args=sft_args,
    train_dataset=sft_train_dataset,
    eval_dataset=sft_eval_dataset,
    processing_class=sft_tokenizer,
    peft_config=lora_config,
)

if sft_use_fp16:
    for param in sft_trainer.model.parameters():
        if param.requires_grad and param.dtype == torch.bfloat16:
            param.data = param.data.to(torch.float16)


sft_trainer.train()

sft_trainer.save_model(SFT_MODEL_DIR)
sft_tokenizer.save_pretrained(SFT_MODEL_DIR)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 982.52it/s]


Train examples: 300
Eval examples: 50
fp16: False
bf16: True


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.538976
20,1.788343
30,1.720310
40,1.544835
50,1.521493
60,1.498727
70,1.541296


('/home/ubuntu/model_sft_qwen_ru/tokenizer_config.json',
 '/home/ubuntu/model_sft_qwen_ru/chat_template.jinja',
 '/home/ubuntu/model_sft_qwen_ru/tokenizer.json')

## 12. Проверка генерации после SFT

In [8]:
final_tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_DIR, trust_remote_code=True)
final_model = AutoModelForCausalLM.from_pretrained(
    SFT_MODEL_DIR,
    torch_dtype='auto',
    device_map='auto',
    trust_remote_code=True,
)

print('SFT Output')
generate_answers(final_model, final_tokenizer, questions_rus, max_new_tokens=256)

Loading weights: 100%|██████████| 336/336 [00:00<00:00, 3972.74it/s]


SFT Output
Model Input 1:
сколько планет в нашей солнечной системе?
Model Output 1:
Солнечная система состоит из 8 планет: Земля, Марс, Венера, Юпитер, Уран, Сатурн, Гемит и Купалец.所有情节 по порядку: Земля, Марс, Венера, Юпитер, Уран, Сатурн, Гемит и Купалец.所有情节 по порядку: Земля, Марс, Венера, Юпитер, Уран, Сатурн, Гемит и Купалец.所有情节 по порядку: Земля, Марс, Венера, Юпитер, Уран, Сатурн, Гемит и Купалец.所有情节 по порядку: Земля, Марс, Венера, Юпитер, Уран, Сатурн, Гемит и Купалец.所有情节 по порядку: Земля, Марс, Венера, Юпитер, Уран, Сатурн, Гемит и Купалец.所有情节 по порядку: Земля, Марс, Венера, Юпитер, Уран, Сатурн
Model Input 2:
расскажи стих
Model Output 2:
Однажды, в одном из тайных мест, я увидел, что у меня есть неизвестная вещь. И я понял, что это нечто важное. И я решил, что это может быть великолепно для меня.actionDate
Model Input 3:
когда собирать крыжовник?
Model Output 3:
Приготовление крыжовника можно приготовить на дно кухонного стола или блюдо в стакане. Достаточно добавит

# Выводы

В рамках проекта был успешно реализован полный цикл создания и обучения языковой модели (LLM), включающий этапы Pretrain и Supervised Fine-Tuning. В качестве базы знаний для предварительного обучения использовался корпус текстов произведений русской классической литературы.

### 1. Обработка данных и токенизация
* Был проведен качественный препроцессинг сырых текстовых данных: очистка, нормализация пунктуации, дедупликация предложений и разбиение текста на смысловые чанки.
* Успешно обучен кастомный **BPE-токенизатор** с размером словаря 3000 токенов, адаптированный специально под лексику русской классической литературы.

### 2. Pretrain
* Инициализирована *decoder-only* языковая модель на базе **LLaMA**.
* Процесс обучения показал устойчивое падение функции потерь (Training/Validation Loss). 
* Промежуточные и финальные результаты генерации на тестовых промптах доказали, что модель успешно усвоила структуру русского языка, грамматику и характерный литературный стиль. Модель научилась связно продолжать заданные фразы, генерируя тексты, имитирующие слог классических произведений.

### 3. SFT 
* Для SFT использовалась модель Qwen/Qwen2.5-0.5B и датасет инструкций `d0rj/alpaca-cleaned-ru`. 
* Финальный скрипт генерации ответов показал, что стадия SFT успешно трансформировала поведение модели: она стала лучше воспринимать запросы пользователя и пытаться формировать структурированные ответы в формате диалога.

## Ограничения
* Размер словаря в 3000 токенов является довольно маленьким, из-за чего многие слова разбиваются на слишком мелкие частицы. 